# EVE vs AdamW — Race to 90% Macro F1

**Task**: ResNet-18 on full CIFAR-10 (50 k train / 10 k test), batch size 128.

Both optimisers use default hyperparameters (`lr=1e-3`).  
EVE runs with `record_diagnostics=False` (pure training, no overhead).  
Training stops when **val macro F1 ≥ 0.90** is first reached, or after `MAX_EPOCHS`.


In [ ]:
import os, sys, subprocess, importlib

BRANCH = "full-batch-probing"

if os.path.exists("/content"):
    repo_dir = "/content/EVE"
    if os.path.exists(repo_dir):
        subprocess.run(["git", "-C", repo_dir, "fetch", "origin"], check=False)
        subprocess.run(["git", "-C", repo_dir, "checkout", BRANCH], check=False)
        subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{BRANCH}"], check=False)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH,
                        "https://github.com/SattamAltwaim/EVE.git", repo_dir], check=False)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    result = subprocess.run(["git", "-C", repo_dir, "log", "--oneline", "-3"],
                            capture_output=True, text=True)
    print("Active commits:\n" + result.stdout.strip())
else:
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if parent not in sys.path:
        sys.path.insert(0, parent)

if "eve_optimizer" in sys.modules:
    importlib.reload(sys.modules["eve_optimizer"])
from eve_optimizer import EVE
print("EVE imported:", EVE)


In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 128
MAX_EPOCHS  = 300          # hard cap if 90% F1 is never reached
TARGET_F1   = 0.90         # race target
LR          = 1e-3
SEED        = 42
NUM_WORKERS = 0

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Config : batch={BATCH_SIZE}  lr={LR}  target_f1={TARGET_F1}  max_epochs={MAX_EPOCHS}")


In [ ]:
# Standard CIFAR-10 augmentation for training; normalise only for val
train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
val_tf = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_ds = torchvision.datasets.CIFAR10(
    root="/tmp/cifar10", train=True,  download=True, transform=train_tf)
val_ds   = torchvision.datasets.CIFAR10(
    root="/tmp/cifar10", train=False, download=True, transform=val_tf)

train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), drop_last=True)
val_dl   = torch.utils.data.DataLoader(
    val_ds,   batch_size=256, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

print(f"Train batches: {len(train_dl)}  |  Val samples: {len(val_ds)}")


In [ ]:
def make_model():
    m = torchvision.models.resnet18(weights=None, num_classes=10)
    return m.to(DEVICE)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    total_loss, preds, targets = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out = model(xb)
        total_loss += loss_fn(out, yb).item() * len(yb)
        preds.extend(out.argmax(1).cpu().tolist())
        targets.extend(yb.cpu().tolist())
    n = len(targets)
    acc     = sum(p == t for p, t in zip(preds, targets)) / n
    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    return total_loss / n, acc, macro_f1


In [ ]:
def train_race(label, make_opt_fn, *, is_eve=False):
    """
    Train until val macro F1 >= TARGET_F1 or MAX_EPOCHS, whichever comes first.
    Returns a history dict.
    """
    torch.manual_seed(SEED)
    model   = make_model()
    loss_fn = nn.CrossEntropyLoss()
    opt     = make_opt_fn(model.parameters())

    hist = dict(train_loss=[], val_loss=[], val_acc=[], val_f1=[],
                epoch_wall_s=[])
    race_epoch, race_time = None, None
    wall_start = time.perf_counter()

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss, n_batches = 0.0, 0

        pbar = tqdm(train_dl, desc=f"Ep {epoch:3d}", leave=False,
                    dynamic_ncols=True)
        for xb, yb in pbar:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out  = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            if is_eve:
                opt.step(model=model, loss_fn=loss_fn, data=(xb, yb),
                         current_loss=loss.item())
            else:
                opt.step()
            epoch_loss += loss.item()
            n_batches  += 1
            pbar.set_postfix(loss=f"{loss.item():.3f}")

        tr_loss = epoch_loss / n_batches
        v_loss, v_acc, v_f1 = evaluate(model, val_dl)
        wall_elapsed = time.perf_counter() - wall_start

        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(v_loss)
        hist["val_acc"].append(v_acc)
        hist["val_f1"].append(v_f1)
        hist["epoch_wall_s"].append(wall_elapsed)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Ep {epoch:3d}/{MAX_EPOCHS}  "
                  f"train_loss={tr_loss:.4f}  "
                  f"val_loss={v_loss:.4f}  "
                  f"val_acc={v_acc:.4f}  "
                  f"val_f1={v_f1:.4f}  "
                  f"[{wall_elapsed/60:.1f} min]")

        if race_epoch is None and v_f1 >= TARGET_F1:
            race_epoch = epoch
            race_time  = wall_elapsed
            print(f"\n  *** {label} hit {TARGET_F1:.0%} macro F1 at epoch {epoch} "
                  f"({race_time/60:.2f} min) ***\n")
            break

    if race_epoch is None:
        best_f1 = max(hist["val_f1"])
        print(f"\n  {label} did not reach {TARGET_F1:.0%} F1 in {MAX_EPOCHS} epochs. "
              f"Best val F1: {best_f1:.4f}")

    hist["race_epoch"] = race_epoch
    hist["race_time_s"] = race_time
    return hist


In [ ]:
results = {}
results["AdamW"] = train_race(
    "AdamW  (lr=1e-3, weight_decay=0.01)",
    lambda p: torch.optim.AdamW(p, lr=LR),
    is_eve=False,
)


In [ ]:
results["EVE"] = train_race(
    "EVE  (K=4, lr=1e-3, rho=0.5, record_diagnostics=False)",
    lambda p: EVE(p, lr=LR, K=4, rho=0.5, record_diagnostics=False),
    is_eve=True,
)


In [ ]:
print("\n" + "="*60)
print("  RACE SUMMARY — first to val macro F1 ≥ {:.0%}".format(TARGET_F1))
print("="*60)
for name, h in results.items():
    if h["race_epoch"] is not None:
        print(f"  {name:8s}  epoch {h['race_epoch']:3d}  "
              f"{h['race_time_s']/60:6.2f} min  "
              f"val_f1={max(h['val_f1']):.4f}")
    else:
        print(f"  {name:8s}  did NOT reach target  "
              f"best_f1={max(h['val_f1']):.4f}")


In [ ]:
COLORS = {"AdamW": "#2196F3", "EVE": "#E91E63"}

def epochs_axis(h):
    return list(range(1, len(h["train_loss"]) + 1))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("EVE vs AdamW — ResNet-18 / Full CIFAR-10",
             fontsize=14, fontweight="bold", y=1.01)

# ── Top-left: train loss comparison ──────────────────────────────────────────
ax = axes[0, 0]
for name, h in results.items():
    ax.plot(epochs_axis(h), h["train_loss"],
            color=COLORS[name], label=name, linewidth=1.5)
ax.set_title("Train Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.legend()
ax.grid(True, alpha=0.3)

# ── Top-right: val loss comparison ───────────────────────────────────────────
ax = axes[0, 1]
for name, h in results.items():
    ax.plot(epochs_axis(h), h["val_loss"],
            color=COLORS[name], label=name, linewidth=1.5)
ax.set_title("Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.legend()
ax.grid(True, alpha=0.3)

# ── Bottom-left: AdamW train vs val ──────────────────────────────────────────
ax = axes[1, 0]
h = results["AdamW"]
ep = epochs_axis(h)
ax.plot(ep, h["train_loss"], color=COLORS["AdamW"], label="Train", linewidth=1.5)
ax.plot(ep, h["val_loss"],   color=COLORS["AdamW"], label="Val",
        linewidth=1.5, linestyle="--")
ax.set_title("AdamW — Train vs Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.legend()
ax.grid(True, alpha=0.3)

# ── Bottom-right: EVE train vs val ───────────────────────────────────────────
ax = axes[1, 1]
h = results["EVE"]
ep = epochs_axis(h)
ax.plot(ep, h["train_loss"], color=COLORS["EVE"], label="Train", linewidth=1.5)
ax.plot(ep, h["val_loss"],   color=COLORS["EVE"], label="Val",
        linewidth=1.5, linestyle="--")
ax.set_title("EVE — Train vs Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.legend()
ax.grid(True, alpha=0.3)

# ── Val macro F1 over time (bonus) ───────────────────────────────────────────
# Add a target line to each loss plot
for name, h in results.items():
    if h["race_epoch"] is not None:
        for row in range(2):
            for col in range(2):
                axes[row, col].axvline(
                    h["race_epoch"], color=COLORS[name],
                    linewidth=0.8, linestyle=":",
                    label=f"{name} reaches 90% F1" if row == 0 and col == 0 else None)

plt.tight_layout()
plt.savefig("race_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved race_results.png")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for name, h in results.items():
    ax.plot(epochs_axis(h), [f * 100 for f in h["val_f1"]],
            color=COLORS[name], label=name, linewidth=1.8)
    if h["race_epoch"] is not None:
        ax.axvline(h["race_epoch"], color=COLORS[name], linewidth=0.8, linestyle=":")

ax.axhline(TARGET_F1 * 100, color="grey", linewidth=1.2,
           linestyle="--", label=f"Target {TARGET_F1:.0%}")
ax.set_title("Val Macro F1 — Race to 90%", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Val macro F1 (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("race_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved race_f1.png")
